# 03 - Text Preprocessing
## ShopEase Europe | Sentiment Analysis Project
**Objective:** Transform cleaned review text into numerical representations 
suitable for machine learning. Covers tokenisation, stop word removal, 
lemmatisation and TF-IDF vectorisation across four languages.

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import nltk
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

# Download required NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# Project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
CLEANED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'clean_reviews.csv')
PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed')

print("Libraries loaded successfully")
print(f"Loading data from: {CLEANED_DATA_PATH}")
print(f"Data exists: {os.path.exists(CLEANED_DATA_PATH)}")

Libraries loaded successfully
Loading data from: c:\Users\User\OneDrive\Desktop\shopease-sentiment-analysis\data\processed\clean_reviews.csv
Data exists: True


## 1.0 Load Cleaned Dataset

In [2]:
# Load the cleaned dataset from notebook 02
df = pd.read_csv(CLEANED_DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nLanguage distribution:")
print(df['language'].value_counts())
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())

Dataset loaded: 120,000 rows x 14 columns

Language distribution:
language
en    52774
fr    24308
de    23203
es    17676
pt     1096
nl      319
af      251
da      228
sv       84
it       59
id        1
no        1
Name: count, dtype: int64

Sentiment distribution:
sentiment
positive    81591
negative    19233
neutral     19176
Name: count, dtype: int64


## 2.0 Tokenisation
Breaking cleaned review text into individual word tokens. 
This is the foundation of all subsequent preprocessing steps.

In [3]:
def tokenise(text):
    # Split on whitespace and punctuation, keep only meaningful tokens
    tokens = str(text).lower().split()
    # Remove tokens that are just punctuation or single characters
    tokens = [t.strip('.,!?;:\'"()[]') for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

# Apply tokenisation
df['tokens'] = df['cleaned_review'].apply(tokenise)

# Verify
print("TOKENISATION COMPLETE")
print(f"\nSample review:  {df['cleaned_review'].iloc[0]}")
print(f"Tokens:         {df['tokens'].iloc[0]}")
print(f"\nAverage tokens per review: {df['tokens'].apply(len).mean():.1f}")

TOKENISATION COMPLETE

Sample review:  solid build, attractive design, works as advertised. love it.
Tokens:         ['solid', 'build', 'attractive', 'design', 'works', 'as', 'advertised', 'love', 'it']

Average tokens per review: 7.8


## 3.0 Stop Word Removal
Stop words are common words that carry no sentiment meaning -
"the", "and", "is", "as". Removing them reduces noise and helps 
the model focus on words that actually signal sentiment.

I apply language-specific stop word lists since the dataset 
contains English, French, German and Spanish reviews.

In [4]:
# Build multilingual stop word lists
english_stops = set(stopwords.words('english'))
french_stops = set(stopwords.words('french'))
german_stops = set(stopwords.words('german'))
spanish_stops = set(stopwords.words('spanish'))

# Map language codes to stop word sets
stop_word_map = {
    'en': english_stops,
    'fr': french_stops,
    'de': german_stops,
    'es': spanish_stops
}

def remove_stopwords(tokens, language):
    stops = stop_word_map.get(language, english_stops)
    return [t for t in tokens if t not in stops]

# Apply stop word removal using detected language
df['tokens_no_stops'] = df.apply(
    lambda row: remove_stopwords(row['tokens'], row['language']), axis=1
)

# Verify
print("STOP WORD REMOVAL COMPLETE")
print(f"\nSample tokens before: {df['tokens'].iloc[0]}")
print(f"Sample tokens after:  {df['tokens_no_stops'].iloc[0]}")
print(f"\nAverage tokens before removal: {df['tokens'].apply(len).mean():.1f}")
print(f"Average tokens after removal:  {df['tokens_no_stops'].apply(len).mean():.1f}")

STOP WORD REMOVAL COMPLETE

Sample tokens before: ['solid', 'build', 'attractive', 'design', 'works', 'as', 'advertised', 'love', 'it']
Sample tokens after:  ['solid', 'build', 'attractive', 'design', 'works', 'advertised', 'love']

Average tokens before removal: 7.8
Average tokens after removal:  5.2


## 4.0 Lemmatisation
Reducing each token to its base dictionary form. This ensures 
"loved", "loving" and "loves" are all treated as the same word 
by the model, reducing vocabulary size and improving generalisation.

spaCy is used for lemmatisation with English as the primary model. 
For non-English reviews, token-level lemmatisation is applied where 
the spaCy model supports it.


In [6]:
# Load spaCy English model
nlp = spacy.load('en_core_web_sm')

def lemmatise(tokens):
    # Join tokens back to string for spaCy processing
    text = ' '.join(tokens)
    doc = nlp(text)
    return [token.lemma_ for token in doc]

print("Applying lemmatisation — please wait...")
df['lemmatised'] = df['tokens_no_stops'].apply(lemmatise)

print("LEMMATISATION COMPLETE")
print(f"\nTokens before: {df['tokens_no_stops'].iloc[0]}")
print(f"Tokens after:  {df['lemmatised'].iloc[0]}")

Applying lemmatisation — please wait...
LEMMATISATION COMPLETE

Tokens before: ['solid', 'build', 'attractive', 'design', 'works', 'advertised', 'love']
Tokens after:  ['solid', 'build', 'attractive', 'design', 'work', 'advertise', 'love']


## 5.0 TF-IDF Vectorisation
Converting lemmatised text to numerical representations for 
classical machine learning models. TF-IDF (Term Frequency-Inverse 
Document Frequency) weights each word by how often it appears in 
a review relative to how common it is across all reviews.

Words that appear frequently in one review but rarely across the 
dataset get higher weights, these are the most discriminating words 
for classification.

In [7]:
# Join lemmatised tokens back to string for TF-IDF input
df['preprocessed_text'] = df['lemmatised'].apply(lambda tokens: ' '.join(tokens))

# Build TF-IDF matrix
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

tfidf_matrix = tfidf.fit_transform(df['preprocessed_text'])

print("TF-IDF VECTORISATION COMPLETE")
print(f"\nVocabulary size:     {len(tfidf.get_feature_names_out()):,} features")
print(f"Matrix shape:        {tfidf_matrix.shape}")
print(f"\nSample features: {list(tfidf.get_feature_names_out()[:20])}")

TF-IDF VECTORISATION COMPLETE

Vocabulary size:     1,155 features
Matrix shape:        (120000, 1155)

Sample features: ['absolument', 'absolutely', 'absolutely love', 'absoluter', 'absoluter betrug', 'absoluto', 'absoluto devolví', 'absoluto lo', 'acceptable', 'acceptable price', 'acceptable quality', 'aceptable', 'aceptable precio', 'achat', 'achat produit', 'achat recommande', 'adore', 'adore merci', 'advertise', 'advertise love']


## 6.0 Save Preprocessed Outputs
Saving the preprocessed dataset and TF-IDF matrix for use 
in the modelling notebook.

In [8]:
import pickle
from scipy.sparse import save_npz

# Save preprocessed dataframe
preprocessed_df_path = os.path.join(PROCESSED_PATH, 'preprocessed_reviews.csv')
df[['review_id', 'cleaned_review', 'preprocessed_text', 
    'sentiment', 'language', 'product_category', 
    'country', 'year', 'month', 'rating']].to_csv(preprocessed_df_path, index=False)

# Save TF-IDF matrix
tfidf_matrix_path = os.path.join(PROCESSED_PATH, 'tfidf_matrix.npz')
save_npz(tfidf_matrix_path, tfidf_matrix)

# Save TF-IDF vectoriser for use during model inference
tfidf_vectoriser_path = os.path.join(PROCESSED_PATH, 'tfidf_vectoriser.pkl')
with open(tfidf_vectoriser_path, 'wb') as f:
    pickle.dump(tfidf, f)

print("PREPROCESSING OUTPUTS SAVED")
print(f"\nPreprocessed dataset: preprocessed_reviews.csv")
print(f"TF-IDF matrix:        tfidf_matrix.npz")
print(f"TF-IDF vectoriser:    tfidf_vectoriser.pkl")
print(f"\nPreprocessed dataset shape: {df.shape[0]:,} rows")
print(f"TF-IDF matrix shape:        {tfidf_matrix.shape}")

PREPROCESSING OUTPUTS SAVED

Preprocessed dataset: preprocessed_reviews.csv
TF-IDF matrix:        tfidf_matrix.npz
TF-IDF vectoriser:    tfidf_vectoriser.pkl

Preprocessed dataset shape: 120,000 rows
TF-IDF matrix shape:        (120000, 1155)


## 7.0 Summary

The cleaned dataset has been fully preprocessed and is ready for 
model training. Starting from raw review text, the pipeline applied 
four sequential transformations:

Tokenisation split each review into individual word tokens, averaging 
7.8 tokens per review. Stop word removal eliminated language-specific 
common words, reducing the average to 5.2 meaningful tokens per review. 
Lemmatisation reduced each token to its base dictionary form using spaCy, 
ensuring word variations are treated consistently by the model. TF-IDF 
vectorisation converted the text to a numerical matrix of 120,000 rows 
by 1,155 features, weighted by term relevance across the corpus.

Three files saved to data/processed/ for downstream modelling:
- preprocessed_reviews.csv - model-ready dataframe with 10 columns
- tfidf_matrix.npz - sparse TF-IDF matrix for classical ML models
- tfidf_vectoriser.pkl - fitted vectoriser for inference on new reviews

Known limitations: a single TF-IDF vectoriser was applied across all 
four languages. A production system would use language-specific 
vectorisers for improved accuracy.